# MAX IV: separate two candidate helicities from multi-energy sums

At each energy the measured target is **I_sum = |F_positive|² + |F_negative|² [+ |B|²]**. Separate measured helicities are held out for diagnostics. Alternate joint two-mode detector/support updates with a physical projection applied to **both** channels:

`log psi(E, ±, r) = C(r) + t(r) [q_charge(E) ± q_magnetic(E) m(r)]`.

This assumes the same sample state, registered object grid, and common illumination across energies and helicities. Unlike the usual primary-plus-background multimode model, both factor-1 channels enter the magnetic model with opposite helicities. With `[1,1,2]`, the third channel B is an incoherent, helicity-independent background with enlarged support; it is excluded from the physical helicity fit. The default thickness is one in the largest support aperture and zero in reference holes; inspect it before interpretation.

This is a constrained separation experiment, not proof of unique recovery. Exchanging all +/− channels leaves every sum unchanged; free magnetic spectra and magnetization also retain sign/scale freedoms. Multiple energies restrict arbitrary mixing but do not guarantee helicity identifiability. Use held-out helicity images to assess separation and label orientation after fitting. No absolute magnetic indices are inferred.


The solver alternates the existing phase-retrieval kernel and physical projector. After warmup, every round performs **joint physical projection of both candidate helicities → phase retrieval against each measured sum**, then repeats using the resulting fields. `MODES=[1,1,2]` adds a nonmagnetic background mode; only the first two modes enter the helicity fit.

`ITERATIONS` controls the initial warmup, `INNER_ITERATIONS` controls phase-retrieval steps between projections, `OUTER_ITERATIONS` controls the number of cycles, and `PROJECTION_RELAXATION` controls the physical update strength. The default warmup is shortened to HAPRE 100 + ER 30, followed by 30 physical/ER cycles with 10 ER iterations each. These are tunable schedules, not established convergence settings.

Keep separate diagnostics after physical projection and after phase retrieval: enforcing the summed intensity does not establish helicity separation. Individual helicities and their difference remain held out. Restart the kernel after changing the shared libraries.

### Hyperspectral redundancy and PRL-inspired initialization

The default starts with a shared positive-real charge seed and a weaker complex magnetic seed across all energies, then transforms these to candidate helicity fields. The initial magnetic field is confined to the material aperture. Positivity is not enforced during retrieval. The existing joint projection fits `log(psi[E,+/-]) = C + t*(qc[E] +/- qm[E]*m)` with a single real spatial magnetization shared across energies. Energy responses are fitted, not inferred from held-out helicities. Every physical projection is followed by the existing summed-intensity phase retrieval. This uses spectral coupling rather than scan overlap, without claiming equivalent identifiability. Set `INITIALIZATION_BASIS="independent_helicity"` for the previous control. `support_projected_history` checks constraint compatibility beyond the final detector projection.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import set_workers
ROOT = Path.cwd()
if (ROOT / "maxiv_phase_test").is_dir(): ROOT = ROOT / "maxiv_phase_test"
sys.path.insert(0, str(ROOT.parent / "library"))
import CCI_core as cci
import fth_phase_workflow as wf
import phase_retrieval_geometry as geometry
import phase_retrieval_universal as pr
import phase_retrieval_universal as universal
from scipy.ndimage import map_coordinates, binary_dilation
from retrieval_progress import retrieval_progress

DATA_ROOT = Path("/media/riccardo/EXT_16TB1/DATA/MAXIV/MaxIV2409")
RAW_FOLDER, AVERAGE_FOLDER = DATA_ROOT / "raw", DATA_ROOT / "proc"
SUPPORT_FILE = ROOT / "processed/Logs/data_recon_ImId_1143_maxiv.hdf5"
OUTPUT_FILE = ROOT / "processed/Logs/data_phase_summed_hyperspectral_2modes_maxiv.hdf5"
# Positive scan, negative scan, averaged dark references, exposure seconds.
# All pairs must describe the same sample state and illumination.
PAIRS = [
    (1152, 1150, (1151, 1153), 0.065),
    (1155, 1157, (1156, 1158), 0.070),
    (1162, 1160, (1159, 1161), 0.065),
    (1167, 1170, (1169, 1171), 0.100),
    (1175, 1173, (1172, 1174), 0.350),
    (1178, 1180, (1177, 1181), 1.000),
    (1185, 1183, (1177, 1181), 1.000),
    (1191, 1189, (1187, 1190), 0.800),
    (1196, 1194, (1193, 1195), 0.600),
    (1204, 1206, (1205, 1203), 0.500),
    (1235, 1232, (1231, 1234), 0.250),
    (1241, 1239, (1237, 1238), 0.250),
]
USE_PHYSICAL_PROJECTION = True
OUTER_ITERATIONS = 30
INNER_ITERATIONS = 10
PROJECTION_RELAXATION = 0.3
REFERENCE_ENERGY_EV = None  # Median energy defines the common reciprocal grid.
ENERGY_TOLERANCE_EV = 0.25
DETECTOR_CENTER = (1137, 1126)
BINNING = 2
CROP = 248
SATURATION = 0.75e4
USE_SAVED_PIXEL_MASK = False
BEAMSTOP_RADIUS = 0  # Detector pixels before crop/binning; 0 adds no circular mask.
FRAME_NORMALIZATION = 1  # Cached images are frame averages; divide by exposure below.
MODES = [1, 1]  # Or [1, 1, 2]: positive, negative, optional nonmagnetic background.
ALGORITHMS = ["HAPRE", "ER"]
ITERATIONS = [100, 30]  # Warmup; increase if needed.
INITIALIZATION_SEED = 0
FFT_WORKERS = 1

INITIALIZATION_BASIS = "charge_magnetic"  # Alternative: "independent_helicity".
MAGNETIC_INITIAL_AMPLITUDE = 0.1  # Amplitude ratio, not intensity weight.


## Load, validate and register each pair
Normalize exposures before summing. Resample centered detector intensities to a common reciprocal grid using the energy ratio and linear interpolation; exclude every interpolation footprint touching an invalid pixel. The Jacobian factor preserves integrated intensity under this coordinate change. The support is assumed to represent the chosen reference-energy object grid. Metadata labels follow the existing MAX IV scan manifest; helicity metadata are retained for inspection.


In [ ]:
def scan_snapshot(scan_id):
    path = RAW_FOLDER / f"2409_maxiv_{scan_id:05d}.h5"
    with h5py.File(path, "r") as handle:
        entry = handle[f"entry{scan_id}"]
        snapshot = entry["measurement/pre_scan_snapshot"]
        def scalar(name):
            if name not in snapshot:
                return None
            values = np.asarray(snapshot[name][()]).reshape(-1)
            if values.size == 0:
                return None
            if np.issubdtype(values.dtype, np.number):
                if not np.allclose(values, values[0]):
                    raise ValueError(f"Scan {scan_id}: {name} changes during acquisition")
                return float(values[0])
            return str(values[0])
        return {key: scalar(key) for key in ("energy", "helicity", "magOOP")}

if len(PAIRS) < 2:
    raise ValueError("Use at least two energy pairs for spectral coupling")
metadata, energies = [], []
for pos_id, neg_id, dark_ids, exposure_s in PAIRS:
    if not np.isfinite(exposure_s) or exposure_s <= 0 or FRAME_NORMALIZATION <= 0 or not dark_ids:
        raise ValueError("Require positive exposure/frame scaling and dark references")
    pair_meta = [scan_snapshot(pos_id), scan_snapshot(neg_id)]
    pair_energy = [m["energy"] for m in pair_meta]
    if any(e is None or not np.isfinite(e) or e <= 0 for e in pair_energy):
        raise ValueError("Missing/invalid energy")
    if abs(pair_energy[0]-pair_energy[1]) > ENERGY_TOLERANCE_EV:
        raise ValueError("Pair energies differ")
    metadata.append(pair_meta); energies.append(np.mean(pair_energy))
fields_meta = [m["magOOP"] for pair in metadata for m in pair]
if any(f is None for f in fields_meta) or not np.allclose(fields_meta, fields_meta[0]):
    raise ValueError("Require matching magnetic-field metadata across all pairs")
energies = np.asarray(energies)
if np.ptp(energies) <= ENERGY_TOLERANCE_EV:
    raise ValueError("Select distinct energies")
reference_energy = float(np.median(energies) if REFERENCE_ENERGY_EV is None else REFERENCE_ENERGY_EV)
if not np.isfinite(reference_energy) or reference_energy <= 0:
    raise ValueError("Invalid reference energy")
print("Energies:", energies, "reference:", reference_energy)
print("Helicity metadata (+, - manifest order):", [[m["helicity"] for m in pair] for pair in metadata])

saved = wf.load_data_dict(SUPPORT_FILE)
support_source = (np.asarray(saved["supportmask"]) != 0).astype(np.uint8)
saved_mask = np.asarray(saved["mask_pixel"]) != 0
if support_source.ndim != 2 or saved_mask.shape != support_source.shape:
    raise ValueError("Support and mask must be matching 2D arrays")
def load_average(scan):
    with h5py.File(AVERAGE_FOLDER / f"scan_{scan:04d}_avg.h5", "r") as handle:
        image = np.asarray(handle["image"][()]).squeeze().astype(float)
    if image.shape != support_source.shape: raise ValueError("Input/support shape mismatch")
    return image

def register_energy(image, excluded, energy):
    scale = reference_energy / energy
    if np.isclose(scale, 1, rtol=0, atol=1e-12):
        return image, excluded
    grid = np.indices(image.shape, dtype=float)
    origin = np.asarray(image.shape) // 2
    coords = origin[:, None, None] + scale * (grid-origin[:, None, None])
    clean = np.where(excluded, 0., image)
    warped = map_coordinates(clean, coords, order=1, mode="constant", cval=0.) * scale**2
    unsafe = binary_dilation(excluded, structure=np.ones((3,3)))
    mask = map_coordinates(unsafe.astype(float), coords, order=1, mode="constant", cval=1.) > 0
    return warped, mask

prepared_pairs, mask_pairs = [], []
for pair_index, (pos_id, neg_id, dark_ids, exposure_s) in enumerate(PAIRS):
    images, masks = [], []
    dark = np.mean([load_average(scan) for scan in dark_ids], axis=0)
    for channel, scan in enumerate((pos_id, neg_id)):
        raw = load_average(scan)
        image = wf.center_image(raw-dark, DETECTOR_CENTER, cci)
        mask = wf.center_image((raw >= SATURATION).astype(np.uint8), DETECTOR_CENTER, cci) != 0
        mask |= ~np.isfinite(image)
        if USE_SAVED_PIXEL_MASK: mask |= saved_mask
        y, x = np.indices(image.shape)
        if BEAMSTOP_RADIUS > 0:
            mask |= np.hypot(y-image.shape[0]//2, x-image.shape[1]//2) < BEAMSTOP_RADIUS
        image = np.nan_to_num(image)/(FRAME_NORMALIZATION*exposure_s)
        image, mask = register_energy(image, mask, metadata[pair_index][channel]["energy"])
        images.append(image.astype(np.float32)); masks.append(mask)
    pair_images, pair_masks, support, _, input_geometry = geometry.prepare(
        np.stack(images), np.stack(masks), support_source,
        dict(crop=CROP, binning=BINNING, roi=None, Nmodes=1))
    prepared_pairs.append(pair_images)
    mask_pairs.append(pair_masks)
    print(f"Prepared energy {pair_index+1}/{len(PAIRS)}",flush=True)
prepared = np.stack(prepared_pairs)
prepared_masks = np.stack(mask_pairs)
del prepared_pairs, mask_pairs
summed_intensity = prepared.sum(axis=1)
mask_pixel = np.any(prepared_masks != 0, axis=1)
mask_pixel |= ~np.isfinite(summed_intensity) | (summed_intensity < 0)
del images, masks
material_thickness = universal.largest_support_component(support).astype(float)
fig, axes = plt.subplots(1, 3, figsize=(12,4))
for ax, img, title in zip(axes, [np.log1p(np.maximum(summed_intensity[0],0)), support, material_thickness],
                         ["First measured sum", "Support", "Assumed material aperture"]):
    ax.imshow(img); ax.set_title(title)
plt.show()


## Alternate physical projection and phase retrieval
Warm up each summed observation using the configured HAPRE/ER schedule. Each outer round then:

1. Fits the existing shared charge/magnetic model to **both** candidate helicity fields across all observations, applying the relaxed physical projection.
2. Uses those projected fields as the starts for the next ER phase-retrieval block against each observation's **summed** intensity and modal supports.
3. Carries those updated fields into the next physical projection.

The first two modes are flattened into coherent +/− observations only while applying `project_fourier_fields_general`; the optional background never enters that magnetic fit. The final returned fields are the result of phase retrieval following the last physical projection. The saved component maps describe the last physical fit, not an exact model representation of the subsequent phase-retrieval fields. Both stages' summed-data residuals are saved.


In [ ]:
if list(MODES) not in ([1, 1], [1, 1, 2]):
    raise ValueError("Choose MODES=[1,1] or [1,1,2]; first two modes are + and - helicity")
if OUTER_ITERATIONS < 1 or INNER_ITERATIONS < 1:
    raise ValueError("Iteration counts must be positive")
nmodes = len(MODES)
mode_labels = ["Candidate +", "Candidate -"] + (["Nonmagnetic background"] if nmodes == 3 else [])
modal_supports = pr.mode_supports(support, MODES, support.shape)
rng = np.random.default_rng(INITIALIZATION_SEED)
def detector_project(fields, intensity, excluded):
    power = np.sum(abs(fields)**2, axis=0)
    valid = ~excluded
    projected = fields.copy()
    nonzero = valid & (power > 0)
    projected[:, nonzero] *= np.sqrt(intensity[nonzero]/power[nonzero])
    zero = valid & (power == 0)
    projected[:, zero] = np.sqrt(intensity[zero]/fields.shape[0])
    return projected

# Reuse the SAME object seeds across energies. The physical projector then
# fits one spatial magnetization with energy-dependent complex responses.
initialization_basis = globals().get("INITIALIZATION_BASIS", "charge_magnetic")
magnetic_initial_amplitude = float(globals().get("MAGNETIC_INITIAL_AMPLITUDE", .1))
if initialization_basis not in ("charge_magnetic", "independent_helicity"):
    raise ValueError("Unknown initialization basis")
if not np.isfinite(magnetic_initial_amplitude) or magnetic_initial_amplitude <= 0:
    raise ValueError("Magnetic initial amplitude must be positive")
charge_seed = support * rng.uniform(.5, 1.5, support.shape)
magnetic_seed = (material_thickness != 0) * (rng.normal(size=support.shape) + 1j*rng.normal(size=support.shape))
magnetic_seed *= magnetic_initial_amplitude * np.linalg.norm(charge_seed) / max(np.linalg.norm(magnetic_seed), 1e-30)
# Unitary change of basis: |plus|²+|minus|² = |charge|²+|magnetic|².
# This is initialization only: no positivity or per-mode power enforcement.
helicity_seed = np.stack([charge_seed + magnetic_seed, charge_seed - magnetic_seed]) / np.sqrt(2.)
initial_fields = []
for target, excluded in zip(summed_intensity, mask_pixel):
    objects = modal_supports * np.exp(1j*rng.uniform(-np.pi, np.pi, (nmodes, *support.shape)))
    if initialization_basis == "charge_magnetic":
        objects[:2] = helicity_seed
        if nmodes == 3:
            objects[2] *= magnetic_initial_amplitude
    f = np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(objects, axes=(-2,-1))), axes=(-2,-1))
    valid = (~excluded) & (target > 0)
    if not np.any(valid): raise ValueError("No positive usable data")
    f *= np.sqrt(target[valid].sum()/np.sum(abs(f[:,valid])**2))
    initial_fields.append(f)
initial_fields = np.stack(initial_fields)
nstages = len(ALGORITHMS)
recipe = pr.default_phase_retrieval_recipe()
recipe.update(
    modes=MODES, algorithm_list=ALGORITHMS, number_iterations=ITERATIONS,
    helicity=["sum"]*nstages, beta_zero=[.5]*nstages,
    beta_mode=["arctan" if mode=="HAPRE" else "const" for mode in ALGORITHMS],
    alpha_zero=[0.]*nstages, alpha_mode=["const"]*nstages,
    RL_its=[0]*nstages, RL_freqs=[10**9]*nstages, TV_freqs=[10**9]*nstages,
    plot_every=[10**9]*nstages, average_img=[1]*nstages,
    Fourier_last=[True]*nstages, output=[True]*nstages,
    Startimage=[initial_fields[0]]+["sum"]*(nstages-1), Startgamma=[None]*nstages,
    crop=0, binning=1, roi=None, hologram_intensity_cutoff_vmin=-1,
    return_format="dict",
)

def update_pair(f, index, warmup=False):
    local = dict(recipe)
    if warmup:
        local["Startimage"] = [f]+["sum"]*(nstages-1)
    else:
        # Build a single ER stage from the existing validated recipe.
        local = {key: ([value[-1]] if isinstance(value, list) and key != "modes" else value)
                 for key, value in recipe.items()}
        local.update(algorithm_list=["ER"], number_iterations=[INNER_ITERATIONS],
                     Startimage=[f], helicity=["sum"], beta_mode=["const"])
    with retrieval_progress(local, kind="unified", leave=False,
                            description=f"Energy {energies[index]:.2f} eV"):
        result = pr.phase_retrieval_algorithm({"sum": summed_intensity[index]},
            mask_pixel=mask_pixel[index], supportmask=support, phase_retrieval_recipe=local)
    return result["full_coherence"]["sum"]

physical_recipe = universal.default_general_phase_retrieval_recipe()
physical_recipe.update(physical_phase_reference=True, energy_values=energies)
flat_energies = np.repeat(energies, 2).tolist()
flat_polarizations = np.tile([1., -1.], len(energies)).tolist()
labels = ["state0"] * len(flat_energies)
beams = ["beam0"] * len(flat_energies)

def physical_project(stack):
    projected, components = universal.project_fourier_fields_general(
        stack[:, :2].reshape(-1, *support.shape), labels, flat_energies, flat_polarizations, beams,
        projection_model="physical_factorized", relaxation=PROJECTION_RELAXATION,
        recipe=physical_recipe, physical_iterations=20,
        material_thickness=np.fft.fftshift(material_thickness),
        projection_supportmask=np.fft.fftshift(support) != 0,
        magnetization_supportmask=np.fft.fftshift(material_thickness) != 0,
        return_components=True)
    output = stack.copy()
    output[:, :2] = projected.reshape(stack.shape[0], 2, *support.shape)
    # Background is helicity-independent and never enters the charge/magnetic fit.
    return output, components

def sum_errors(stack):
    prediction = np.sum(abs(stack)**2, axis=1)
    return np.asarray([np.linalg.norm((pred-target)[~mask])/max(np.linalg.norm(target[~mask]),1e-30)
                       for pred,target,mask in zip(prediction,summed_intensity,mask_pixel)])

def support_projected_sum_errors(stack):
    objects = np.fft.ifftshift(np.fft.fft2(np.fft.fftshift(stack, axes=(-2,-1))), axes=(-2,-1))
    objects *= modal_supports[None]
    supported = np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(objects, axes=(-2,-1))), axes=(-2,-1))
    return sum_errors(supported)

history, retrieval_history, components = [], [], None
support_history = []
with set_workers(FFT_WORKERS):
    fields = np.stack([update_pair(f, i, warmup=True) for i,f in enumerate(initial_fields)])
    warmup_fields = fields.copy()
    for outer in range(OUTER_ITERATIONS):
        if USE_PHYSICAL_PROJECTION:
            fields, components = physical_project(fields)
        # Preserve the last physical-stage fields separately from the subsequent PR.
        physical_fields = fields.copy()
        history.append(sum_errors(physical_fields))
        fields = np.stack([update_pair(f, i) for i,f in enumerate(physical_fields)])
        retrieval_history.append(sum_errors(fields))
        support_history.append(support_projected_sum_errors(fields))
        print(f"Round {outer+1}/{OUTER_ITERATIONS}: physical residual {np.mean(history[-1]):.5g}, "
              f"after phase retrieval {np.mean(retrieval_history[-1]):.5g}, "
              f"after support {np.mean(support_history[-1]):.5g}",flush=True)
pre_detector_fields = physical_fields  # Compatibility name for last physical-stage output.


## Validate separation after the alternating cycles
Compare the original, held-out helicity difference to the predicted difference, reporting both global assignments without per-energy swaps or rescaling. Identical candidate helicities predict zero difference and give relative difference error **1**. Plot the errors after physical projection and after phase retrieval separately; a low final sum residual can simply reflect the Fourier constraint.

Component maps belong to the last physical fit. Final fields belong to the following phase-retrieval block. The original measured individual helicities are never separate retrieval targets. With three modes, validation assumes half the background intensity in each helicity; that equal allocation cancels from their difference.


In [ ]:
prediction = np.sum(abs(fields)**2, axis=1)
modal_intensities = abs(fields)**2
print("Last physical-stage sum errors:", sum_errors(physical_fields))
print("Final phase-retrieval sum errors:", sum_errors(fields))
background_intensity = (modal_intensities[:, 2] if nmodes == 3 else np.zeros_like(prediction))
helicity_intensities = modal_intensities[:, :2] + background_intensity[:, None]/2
print("Modal power fractions per energy:", modal_intensities.sum(axis=(-2,-1))/np.maximum(prediction.sum(axis=(-2,-1))[:,None],1e-30))
heldout_errors = np.empty((2, len(energies)))
for swap in (0,1):
    candidates = helicity_intensities[:, ::-1] if swap else helicity_intensities
    for i in range(len(energies)):
        valid = np.broadcast_to(~mask_pixel[i], prepared[i].shape)
        heldout_errors[swap,i] = np.linalg.norm((candidates[i]-prepared[i])[valid])/max(np.linalg.norm(prepared[i][valid]),1e-30)
print("Held-out errors (direct, globally swapped):", heldout_errors)
print("Assignment remains unanchored during retrieval; diagnostic best global swap:", int(np.argmin(np.mean(heldout_errors,axis=1))))
predicted_difference = helicity_intensities[:,0] - helicity_intensities[:,1]
measured_difference = prepared[:,0] - prepared[:,1]
difference_errors = np.asarray([
    [np.linalg.norm((sign*pred-obs)[~mask])/max(np.linalg.norm(obs[~mask]),1e-30)
     for pred,obs,mask in zip(predicted_difference,measured_difference,mask_pixel)]
    for sign in (1,-1)])
print("Held-out helicity-difference errors (direct, global swap):", difference_errors)
exit_waves = np.fft.ifftshift(np.fft.fft2(np.fft.fftshift(fields,axes=(-2,-1))),axes=(-2,-1))
roi = geometry.support_bounding_roi(support)
fig, axes = plt.subplots(nmodes,3,figsize=(12,3.5*nmodes))
for mode in range(nmodes):
    axes[mode,0].imshow(abs(exit_waves[0,mode])[roi],cmap="gray")
    axes[mode,1].imshow(np.angle(exit_waves[0,mode])[roi],cmap="twilight",vmin=-np.pi,vmax=np.pi)
    axes[mode,2].imshow(np.log1p(modal_intensities[0,mode]),cmap="magma")
    for ax,title in zip(axes[mode],["Object amplitude", "Object phase", "Predicted intensity"]):
        ax.set_title(f"{mode_labels[mode]}: {title}")
fig.tight_layout()
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].semilogy(np.arange(1,len(history)+1), np.maximum(np.mean(history,axis=1),1e-16),label="After physical projection")
axes[0].semilogy(np.arange(1,len(retrieval_history)+1), np.maximum(np.mean(retrieval_history,axis=1),1e-16),label="After phase retrieval")
axes[0].legend()
axes[0].set(xlabel="Physical / phase-retrieval cycle", ylabel="Mean summed-intensity relative error")
for swap in (0,1): axes[1].plot(energies,difference_errors[swap],"o",label=["Direct","Global swap"][swap])
axes[1].set(xlabel="Energy (eV)",ylabel="Held-out difference relative error"); axes[1].legend()
axes[1].axhline(1.,color="black",linestyle="--",label="No separation")
axes[1].legend()
plt.show()

# Show the strongest measured contrast for inspection only, never for fitting/selection.
contrast_norm=np.array([np.linalg.norm(d[~mask]) for d,mask in zip(measured_difference,mask_pixel)])
inspect_index=int(np.argmax(contrast_norm))
valid=~mask_pixel[inspect_index]
obs=measured_difference[inspect_index]
pred=predicted_difference[inspect_index]
limit=np.percentile(abs(obs[valid]),99)
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,img,title in zip(axes,[obs,pred,pred-obs],["Measured difference (held out)","Recovered difference (+/- convention)","Difference residual"]):
    ax.imshow(np.ma.masked_where(~valid,img),cmap="RdBu_r",vmin=-limit,vmax=limit);ax.set_title(title)
fig.tight_layout();plt.show()
print("Equal-helicity / zero-difference baseline error: 1.0")
print("Difference norm ratio:",np.linalg.norm(pred[valid])/max(np.linalg.norm(obs[valid]),1e-30))
print("Difference correlation:",np.corrcoef(obs[valid],pred[valid])[0,1])

if components is not None:
    fig,axes=plt.subplots(1,3,figsize=(14,4))
    axes[0].imshow(np.fft.ifftshift(components["magnetization"][0])[roi],cmap="RdBu_r",vmin=-1,vmax=1);axes[0].set_title("Shared fitted magnetization")
    for ax,key in zip(axes[1:],["charge_response","magnetic_response"]):
        ax.plot(components["energy_names"],components[key].real,"o-",label="Real")
        ax.plot(components["energy_names"],components[key].imag,"o-",label="Imaginary")
        ax.set(xlabel="Energy (eV)",title=key);ax.legend()
    plt.show()

valid_all=~mask_pixel
truth_norm=np.linalg.norm(measured_difference[valid_all])
global_difference_errors=np.array([
    np.linalg.norm((sign*predicted_difference-measured_difference)[valid_all])/max(truth_norm,1e-30)
    for sign in (1,-1)])
separation_summary=dict(global_difference_errors=global_difference_errors.tolist(),
    zero_difference_baseline=1.0,completed_cycles=OUTER_ITERATIONS,
    beats_zero_difference=bool(truth_norm>0 and np.min(global_difference_errors)<1.0))
print("Separation summary:",separation_summary)
if not separation_summary["beats_zero_difference"]:
    print("NO demonstrated separation: this run does not improve on predicting identical helicities.")
else:
    print("Difference prediction improves on equality; this alone does not prove unique recovery.")

# Unlike final Fourier residuals, this tests support/data compatibility.
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(1, len(support_history)+1), np.mean(support_history, axis=1), marker="o")
ax.set(xlabel="Physical / retrieval cycle", ylabel="Support-projected sum error")
ax.grid(alpha=.3)
plt.show()


In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True,exist_ok=True)
with h5py.File(OUTPUT_FILE,"w") as out:
    out.attrs["workflow"] = "summed_multi_energy_candidate_helicity_physical_projection"
    out.attrs["pairs_json"] = json.dumps(PAIRS)
    out.attrs["scan_metadata_json"] = json.dumps(metadata)
    out.attrs["mode_assignment"] = "candidate +,-; global helicity exchange unresolved"
    out.attrs["mode_labels_json"] = json.dumps(mode_labels)
    out.attrs["background_convention"] = "one summed nuisance mode; half intensity per helicity for validation"
    out.attrs["solver"]="alternating_physical_projection_then_phase_retrieval"
    out.attrs["separation_summary_json"]=json.dumps(separation_summary)
    out.attrs["settings_json"] = json.dumps(dict(seed=INITIALIZATION_SEED, modes=MODES,
        initialization_basis=initialization_basis, magnetic_initial_amplitude=magnetic_initial_amplitude,
        crop=CROP, binning=BINNING, frame_normalization=FRAME_NORMALIZATION,
        reference_energy_ev=reference_energy, physical_projection=USE_PHYSICAL_PROJECTION,
        outer_iterations=OUTER_ITERATIONS, inner_iterations=INNER_ITERATIONS,
        projection_relaxation=PROJECTION_RELAXATION, algorithms=ALGORITHMS, iterations=ITERATIONS,
        support_file=str(SUPPORT_FILE), detector_center=DETECTOR_CENTER,
        saturation=SATURATION, use_saved_mask=USE_SAVED_PIXEL_MASK, beamstop_radius=BEAMSTOP_RADIUS))
    for key,value in dict(fields=fields,initial_fields=initial_fields,warmup_fields=warmup_fields,
        pre_detector_fields=pre_detector_fields,exit_waves=exit_waves,
        measured_polarizations=prepared,summed_intensity=summed_intensity,
        mask_pixel=mask_pixel,supportmask=support,material_thickness=material_thickness,
        predicted_intensity=prediction,energies_ev=energies,history=history,retrieval_history=retrieval_history,
        support_projected_history=support_history,
        heldout_errors=heldout_errors, difference_errors=difference_errors,
        helicity_intensities=helicity_intensities, background_intensity=background_intensity,
        predicted_difference=predicted_difference, modal_supportmask=modal_supports).items():
        out.create_dataset(key,data=value,compression="gzip")
    if components is not None:
        group = out.create_group("last_physical_fit")
        group.attrs["note"] = "Last physical fit before the final phase-retrieval block; not an exact model of final fields"
        for key in ("common_log_objects", "charge_response", "magnetic_response", "magnetization", "material_thickness"):
            if key in components: group.create_dataset(key,data=components[key],compression="gzip")
print("Saved:",OUTPUT_FILE)


## Check basis ambiguity before interpreting separation

Run [07_maxiv_modal_subspace_validation.ipynb](07_maxiv_modal_subspace_validation.ipynb) for independent coherent references and a blind two-mode control with optimal constant-unitary alignment. The raw mode/swap plots above test the assumed helicity basis, not recovery of an arbitrary equivalent modal basis. The alternating physical-projection loop here is unchanged.

With `Fourier_last=True`, post-retrieval detector residuals are small by construction. Notebook 07 also checks the detector residual after object support projection.
